
# Inference & evaluation — hadron→parton posterior over groomed Lund trees

A **standalone** notebook that loads a trained `h2p-rsd-junipr` checkpoint and
demonstrates, with figures, how well the learned posterior

$$q_\phi(y\mid x)\quad\text{over the groomed parton-level Lund tree }y$$

recovers the **truth** $y$ from a hadron-level (groomed) jet $x$. It exercises the
three things every model family implements — `log_prob`, `sample` (posterior
draws), and `map_estimate` (the **best point estimate**, $\hat y=\arg\max_y q_\phi(y\mid x)$) —
and contrasts them against the *plain-RSD* baseline (treat the hadron-level $x$ as
if it were the parton truth).

**Inputs** (set in the *Parameters* cell):

| Parameter | Meaning |
|---|---|
| `CKPT_PATH` | a trained `best.ckpt`; `None` → auto-discover the newest `runs/*/best.ckpt` |
| `ROOT_PATH` | a test `jets.root` RNTuple; `None` → generate **synthetic** matched test data so the notebook is fully self-contained |

Every jet here is a *matched pair* $(x, y)$: the hadron-level sequence $x$ is the
network input, and the parton-level sequence $y$ is the truth we score against. The
figures fall into two groups, as requested:

1. **Best point estimate vs. truth** — MAP multiplicity & leading-emission recovery,
   coordinate marginals, single-jet MAP tree.
2. **Posterior vs. truth** — credible bands, SBC / PIT, and coverage calibration.


## 0. Parameters

Edit these and re-run. Defaults are a fully standalone synthetic example.

In [ ]:
# --- inputs -----------------------------------------------------------------
CKPT_PATH   = None      # path to a best.ckpt; None -> newest runs/*/best.ckpt
ROOT_PATH   = None      # path to a test jets.root; None -> synthetic test data
#                         (try e.g. "cpp/build/jets.root" to use real PYTHIA jets)
NTUPLE_NAME = "Jets"    # RNTuple name inside the ROOT file

# --- test sample ------------------------------------------------------------
N_TEST_JETS = 2000      # synthetic jets to generate when ROOT_PATH is None
SEED        = 1234      # seeds data, sampling and figures (reproducible)
DEVICE      = "cpu"     # "cpu" is deterministic for inference; "auto"/"mps"/"cuda" also OK

# --- inference knobs --------------------------------------------------------
N_POSTERIOR       = 600   # posterior draws for the single-jet showcase
N_CLOSURE         = 250   # jets in the aggregate closure / calibration loop
N_CLOSURE_SAMPLES = 250   # posterior draws per jet inside that loop
SHOWCASE_JET      = None  # index for the single-jet deep-dive (None -> auto-pick)
LENGTH_FLOOR_QUANTILE = 0.15  # per-jet MAP floor at this quantile of P(n|x); 0.0 -> off
MBR_BACKEND       = "pot"  # MBR optimal-transport backend: "pot" (default) | "energyflow" | "surrogate"
MBR_N_CANDIDATES  = 24     # MBR candidate cap per jet (0 = all K draws; smaller = faster, O(cand*K);
#                            MBR_BACKEND="surrogate" is a fast, OT-free first pass)

## 1. Imports, helpers & house style

In [ ]:
import csv
import math
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from omegaconf import OmegaConf

from h2p_rsd_junipr.geometry import Geometry
from h2p_rsd_junipr.features import node_raw
from h2p_rsd_junipr.models.base import build_model
from h2p_rsd_junipr.train.checkpoint import load_for_inference
from h2p_rsd_junipr.train.trainer import seed_everything, select_device
from h2p_rsd_junipr.data.rntuple import load_rntuple
from h2p_rsd_junipr.data.synthetic import synthetic_matched_dataset
from h2p_rsd_junipr.data.dataset import MatchedLundDataset, collate
from h2p_rsd_junipr.eval.closure import (
    leading_emission_cell, lund_distance, lund_tree_str,
)
from h2p_rsd_junipr.inference.length import learned_min_emissions

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 12, "figure.titlesize": 13,
})

# consistent colour language used across every figure
C_TRUTH = "#2ca02c"   # truth y          (green)
C_MAP   = "#d62728"   # MAP point estimate (red)
C_POST  = "#1f77b4"   # posterior        (blue)
C_RSD   = "#7f7f7f"   # plain-RSD x baseline (grey)


def repo_root(start: Path | None = None) -> Path:
    """Walk up from the notebook to the directory holding pyproject.toml."""
    p = (start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / "pyproject.toml").exists():
            return cand
    return p


def find_latest_checkpoint(runs_dir: Path) -> Path | None:
    """Newest runs/*/best.ckpt (falls back to last.ckpt) by modification time."""
    cks = sorted(runs_dir.glob("*/best.ckpt"), key=lambda q: q.stat().st_mtime)
    if not cks:
        cks = sorted(runs_dir.glob("*/last.ckpt"), key=lambda q: q.stat().st_mtime)
    return cks[-1] if cks else None


def resolve_device(name: str) -> torch.device:
    if name == "auto":
        return select_device()        # cuda > mps > cpu
    return torch.device(name)


REPO = repo_root()
print("repo root:", REPO)


## 2. Load the trained model

We rebuild the model **from the checkpoint's own config snapshot** (so the
geometry, encoder and heads always match the trained weights), then put it in
`eval()` mode. This is the explicit form of `serving.api.load_service_model`.


In [ ]:
ckpt = Path(CKPT_PATH) if CKPT_PATH else find_latest_checkpoint(REPO / "runs")
assert ckpt is not None and ckpt.exists(), (
    "No checkpoint found. Train one with `h2p-rsd-junipr train ...` "
    "or set CKPT_PATH explicitly."
)
device = resolve_device(DEVICE)
seed_everything(SEED, deterministic=True)

info  = load_for_inference(str(ckpt), map_location=device)
cfg   = OmegaConf.create(info["config"])
geom  = Geometry.from_config(cfg.geometry)
model = build_model(cfg, geom).to(device)
model.load_state_dict(info["model_state"])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"checkpoint : {ckpt.relative_to(REPO)}")
print(f"model      : {info['model_name']}   encoder={cfg.encoder.name}   "
      f"continuous_coords={cfg.model.continuous_coords}")
print(f"geometry   : n_bins={geom.n_bins}  n_cells={geom.n_cells}  "
      f"ln(1/dR)∈{geom.ln_invdelta_range}  ln kt∈{geom.ln_kt_range}")
print(f"parameters : {n_params/1e3:.1f}k   device={device}")

# best validation NLL/jet from the run's training curve, if available
mcsv = ckpt.parent / "metrics.csv"
if mcsv.exists():
    rows = list(csv.DictReader(mcsv.open()))
    col = next((c for c in rows[-1] if "val" in c.lower() and "nll" in c.lower()), None)
    if col:
        best = min(float(r[col]) for r in rows if r.get(col))
        print(f"train curve: best val NLL/jet = {best:.3f}  ({len(rows)} logged steps)")
        TRAIN_BEST_NLL = best
    else:
        TRAIN_BEST_NLL = None
else:
    TRAIN_BEST_NLL = None


## 3. Load the test data — ROOT file **or** synthetic

If `ROOT_PATH` points at a readable `jets.root` we use it; otherwise we fall back
to the in-repo synthetic matched simulator so the notebook runs anywhere. Either
way we get a list of per-jet dicts with a hadron-level `x` and a parton-level
**truth** `y`, then wrap them in `MatchedLundDataset` (which discretises `y` onto
the model's Lund-cell geometry). The encoder needs a non-empty input and closure
needs truth, so we keep only jets with at least one node on **both** sides.


In [ ]:
jets, source_desc = None, None
if ROOT_PATH:
    rp = Path(ROOT_PATH)
    rp = rp if rp.is_absolute() else (REPO / rp)
    jets = load_rntuple(str(rp), NTUPLE_NAME)
    if jets:
        source_desc = f"ROOT RNTuple  {rp.name}:{NTUPLE_NAME}"
if jets is None:
    jets = synthetic_matched_dataset(N_TEST_JETS, seed=SEED)
    source_desc = f"synthetic matched simulator  (n={N_TEST_JETS}, seed={SEED})"

# encoder needs a non-empty x; closure needs a non-empty truth y -> drop empties
n_raw = len(jets)
jets = [j for j in jets
        if len(np.asarray(j["x"][0])) >= 1 and len(np.asarray(j["y"][0])) >= 1]
if n_raw != len(jets):
    print(f"(dropped {n_raw - len(jets)} jets with an empty x or y sequence)")
ds = MatchedLundDataset(jets, geom)

gens = sorted({j.get("generator", "?") for j in jets})
mult_x = np.array([len(j["x"][0]) for j in jets])
mult_y = np.array([len(j["y"][0]) for j in jets])
print(f"source     : {source_desc}")
print(f"generator  : {', '.join(map(str, gens))}")
print(f"jets       : {len(jets)} with truth")
print(f"mean mult. : hadron x = {mult_x.mean():.2f}    parton truth y = {mult_y.mean():.2f}")
assert len(ds) > 10, "need >10 matched jets for the evaluation"


## 4. Likelihood sanity check — $-\log q_\phi(y\mid x)$ over the test set

The exact per-jet NLL is vectorised over a batch. Its mean should sit near the
**best validation NLL** the model reached in training (dashed line); a heavy right
tail flags jets the posterior explains poorly.


In [ ]:
B = min(512, len(ds))
batch = collate([ds[i] for i in range(B)])
batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
with torch.inference_mode():
    nll = (-model.log_prob(batch)).cpu().numpy()

fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.hist(nll, bins=40, color=C_POST, alpha=0.85, edgecolor="white", linewidth=0.4)
ax.axvline(nll.mean(), color="k", lw=2,
           label=f"test mean = {nll.mean():.2f}")
if TRAIN_BEST_NLL is not None:
    ax.axvline(TRAIN_BEST_NLL, color=C_MAP, ls="--", lw=2,
               label=f"train best val = {TRAIN_BEST_NLL:.2f}")
ax.set_xlabel(r"$-\log q_\phi(y\mid x)$  per jet")
ax.set_ylabel("jets")
ax.set_title(f"Per-jet negative log-likelihood  (n={B})")
ax.legend()
fig.tight_layout(); plt.show()
print(f"mean NLL/jet = {nll.mean():.3f}   median = {np.median(nll):.3f}   "
      f"90th pct = {np.percentile(nll, 90):.3f}")


## 5. Single-jet showcase — posterior **and** best point estimate vs. truth

One jet, examined closely. **Left:** the posterior over multiplicity (number of
primary splittings) with the MAP, the posterior mean, the 68% credible region and
the **truth** marked. **Right:** the posterior density on the Lund plane (cell
centres of all draws) with the MAP tree (red ⋆, connected as the primary
"caterpillar"), the truth nodes (green ●) and the plain-RSD hadron-level nodes
(grey ✕) overlaid.


In [ ]:
def pick_showcase(prefer_mult=3):
    if SHOWCASE_JET is not None:
        return int(SHOWCASE_JET)
    mults = np.array([int(ds[i]["ny"]) for i in range(len(ds))])
    cand = np.where(mults >= prefer_mult)[0]
    return int(cand[0]) if len(cand) else int(mults.argmax())

jet_i = pick_showcase()
item  = ds[jet_i]
xf = item["xf"].unsqueeze(0).to(device)
nx = torch.tensor([item["nx"]], device=device)

torch.manual_seed(SEED)
draws = model.sample(xf, nx, n=N_POSTERIOR)
mp    = model.map_estimate(xf, nx)

mult     = np.array([len(d) for d in draws])
y_truth  = item["yraw"].numpy()                 # (ny, 4): ln(1/dR), ln kt, ln z, psi
x_raw    = node_raw(*jets[jet_i]["x"])          # plain-RSD hadron-level nodes
n_true   = int(item["ny"])
cr68     = np.percentile(mult, [16, 84])

# leading-emission Lund distance to truth: MAP vs plain-RSD
ly   = leading_emission_cell(item["yc"].tolist(), geom)
d_map = lund_distance(leading_emission_cell([n.cell for n in mp.nodes], geom), ly, geom)
d_rsd = lund_distance(
    leading_emission_cell(geom.seq_cells(*jets[jet_i]["x"][:2]).tolist(), geom), ly, geom)

print(f"showcase jet #{jet_i}")
print(f"  multiplicity : truth={n_true}   MAP={mp.multiplicity}   "
      f"plain-RSD x={len(x_raw)}   posterior={mult.mean():.2f}±{mult.std():.2f} "
      f"(68% CR [{cr68[0]:.0f}, {cr68[1]:.0f}])")
print(f"  leading-emission Lund distance to truth : MAP={d_map:.3f}   "
      f"plain-RSD={d_rsd:.3f}   (lower is better)")
print(f"  log q(y_hat|x) = {mp.logprob:.3f}")

In [ ]:
fig, (axm, axl) = plt.subplots(1, 2, figsize=(13, 4.6))

# -- (a) posterior multiplicity ------------------------------------------------
bins = np.arange(0, max(mult.max(), n_true, mp.multiplicity) + 2) - 0.5
axm.hist(mult, bins=bins, color=C_POST, alpha=0.8, edgecolor="white",
         label=f"posterior draws (n={N_POSTERIOR})")
axm.axvspan(cr68[0], cr68[1], color=C_POST, alpha=0.15, label="68% CR")
axm.axvline(mp.multiplicity, color=C_MAP, ls="--", lw=2, label=f"MAP = {mp.multiplicity}")
axm.axvline(mult.mean(), color=C_POST, ls=":", lw=2, label=f"post. mean = {mult.mean():.2f}")
axm.axvline(n_true, color=C_TRUTH, lw=2.5, label=f"truth = {n_true}")
axm.set_xlabel("multiplicity  (# primary splittings)")
axm.set_ylabel("draws")
axm.set_title("(a) posterior over multiplicity")
axm.legend(fontsize=9)

# -- (b) Lund plane ------------------------------------------------------------
pts = np.array([geom.cell_center(c) for d in draws for c in d]) if any(draws) else np.zeros((0, 2))
if len(pts):
    h = axl.hist2d(pts[:, 0], pts[:, 1], bins=geom.n_bins,
                   range=[list(geom.ln_invdelta_range), list(geom.ln_kt_range)],
                   cmap="Blues", cmin=1)
    cb = fig.colorbar(h[3], ax=axl); cb.set_label("posterior draws / cell")

mu, mv = [n.ln_invDelta for n in mp.nodes], [n.ln_kt for n in mp.nodes]
axl.plot(mu, mv, "-", color=C_MAP, lw=1.2, alpha=0.7, zorder=4)
axl.scatter(mu, mv, marker="*", s=240, color=C_MAP, edgecolor="k",
            linewidth=0.5, zorder=5, label="MAP tree")
axl.scatter(y_truth[:, 0], y_truth[:, 1], marker="o", s=80, facecolor="none",
            edgecolor=C_TRUTH, linewidth=2, zorder=6, label="truth y")
axl.scatter(x_raw[:, 0], x_raw[:, 1], marker="x", s=60, color=C_RSD,
            linewidth=1.6, zorder=3, label="plain-RSD x")
axl.set_xlim(*geom.ln_invdelta_range); axl.set_ylim(*geom.ln_kt_range)
axl.set_xlabel(r"$\ln\,1/\Delta R$"); axl.set_ylabel(r"$\ln\,k_t$")
axl.set_title("(b) posterior density on the Lund plane")
axl.legend(loc="upper left", fontsize=9, framealpha=0.9)

fig.suptitle(f"Single-jet posterior & MAP vs truth  —  jet #{jet_i}")
fig.tight_layout(); plt.show()

In [ ]:
# The same jet as text: MAP tree vs plain-RSD vs truth (dLund column = per-node
# Lund-plane distance to the aligned truth node).
print(lund_tree_str(mp,     "model MAP groomed shower",      geom, ref=item["yraw"]))
print()
print(lund_tree_str(x_raw,  "plain-RSD groomed shower (x)",  geom, ref=item["yraw"]))
print()
print(lund_tree_str(item["yraw"], "true groomed shower",     geom))


## 6. Aggregate closure vs. truth (many jets)

Loop over `N_CLOSURE` held-out jets, computing the MAP and a posterior for each.
Because there is no per-node $x\leftrightarrow y$ correspondence, we use
**node-alignment-free** observables: multiplicity and the *leading-emission*
(hardest-$k_t$) splitting. Everything downstream is built from this one pass.


In [ ]:
N = min(N_CLOSURE, len(ds))
torch.manual_seed(SEED)

rec = {k: [] for k in (
    "n_true", "n_map", "n_map_raw", "n_map_floor", "n_mbr", "n_x", "post_mean", "post_median",
    "d_mode", "d_x",
    "rank", "pit", "cover68",
)}
post_mults, map_coords, truth_coords, x_coords = [], [], [], []

for i in range(N):
    item = ds[i]
    xf = item["xf"].unsqueeze(0).to(device)
    nx = torch.tensor([item["nx"]], device=device)
    y_cells = item["yc"].tolist()
    ny = len(y_cells)
    x_cells = geom.seq_cells(*jets[i]["x"][:2]).tolist()

    mp     = model.map_estimate(xf, nx)                    # floored (default min_emissions=1)
    mp_raw = model.map_estimate(xf, nx, min_emissions=0)   # unfloored: the historical collapse
    draws  = model.sample(xf, nx, n=N_CLOSURE_SAMPLES)
    mults  = np.array([len(d) for d in draws])
    # learned per-jet floor: reuse the draws above (no second sample) to read the
    # quantile of P(n|x), then re-decode the MAP under that floor
    eff      = learned_min_emissions(model, xf, nx, quantile=LENGTH_FLOOR_QUANTILE,
                                     base_floor=1, mults=mults)
    mp_floor = model.map_estimate(xf, nx, min_emissions=eff)
    # MBR (perturbative Lund): the drawn tree of least expected Lund-EMD to the posterior,
    # reusing the same draws — mode-free and floor-free (min_emissions has no effect on it)
    mbr = model.map_or_mbr(xf, nx, draws=draws, point_estimator='mbr',
                           mbr_backend=MBR_BACKEND, mbr_n_candidates=MBR_N_CANDIDATES)

    ly = leading_emission_cell(y_cells, geom)
    lead = [c for c in (leading_emission_cell(d, geom) for d in draws) if c is not None]
    if ly is None or not lead:
        continue

    vals, counts = np.unique(np.array(lead), return_counts=True)
    mode_cell = int(vals[counts.argmax()])
    rec["d_mode"].append(lund_distance(mode_cell, ly, geom))
    rec["d_x"].append(lund_distance(leading_emission_cell(x_cells, geom), ly, geom))

    rec["n_true"].append(ny)
    rec["n_map"].append(mp.multiplicity)
    rec["n_map_raw"].append(mp_raw.multiplicity)
    rec["n_map_floor"].append(mp_floor.multiplicity)
    rec["n_mbr"].append(mbr.multiplicity)
    rec["n_x"].append(len(x_cells))
    rec["post_mean"].append(float(mults.mean()))
    rec["post_median"].append(float(np.median(mults)))
    post_mults.append(mults)

    # SBC rank of the true multiplicity among draws (Talts et al. 2018) + PIT
    rec["rank"].append((np.sum(mults < ny) + 0.5 * np.sum(mults == ny)) / max(len(mults), 1))
    rec["pit"].append(float(np.mean(mults <= ny)))

    # 68% HPD coverage of the true leading-emission cell
    order = np.argsort(-counts)
    cum = np.cumsum(counts[order]) / counts.sum()
    k68 = int(np.searchsorted(cum, 0.68)) + 1
    rec["cover68"].append(1.0 if ly in set(int(c) for c in vals[order][:k68]) else 0.0)

    map_coords.append(np.array([[n.ln_invDelta, n.ln_kt, n.ln_z, n.psi] for n in mp.nodes])
                      if mp.nodes else np.zeros((0, 4)))
    truth_coords.append(item["yraw"].numpy())
    x_coords.append(node_raw(*jets[i]["x"]))

rec = {k: np.array(v) for k, v in rec.items()}
print(f"evaluated {len(rec['n_true'])} / {N} jets "
      f"(K={N_CLOSURE_SAMPLES} posterior draws each)")


### 6a. Multiplicity recovery — three point estimators

How well a single **point estimate** counts splittings. Perfect recovery lies on the
diagonal; legends report the signed bias $\langle n-n_\text{true}\rangle$.

Five estimators are compared because they behave very differently here:

- **MAP** $\hat y=\arg\max_y q_\phi(y\mid x)$ — the *joint mode*. For a discrete
  autoregressive posterior this is **length-biased**: each emission pays the
  100-way cell head's entropy, while "stop" costs a roughly fixed amount, so for the
  harder (higher-multiplicity) jets the single most-probable explicit tree loses to
  the degenerate empty tree — the un-floored argmax then collapses to $n=0$. *The
  model is not saying the jet is empty* — at the first step $p_\text{cont}\approx1$.
  The decoder fixes this with a **minimum-emission floor** (`decode.min_emissions=1`,
  the default), so the MAP shown here never returns 0; the annotation reports the
  collapse fraction the *un-floored* MAP (`min_emissions=0`) would have suffered.
- **MAP (learned floor)** — the same beam-search MAP, but floored *per jet* at the
  $\alpha=$`LENGTH_FLOOR_QUANTILE` quantile of the model's own length belief
  $P(n\mid x)$ (here $\alpha=0.15$): $\hat n=\max(\texttt{min\_emissions},\,
  Q_\alpha(P(n\mid x)))$. This replaces the hard *global* floor with a learned,
  per-jet lower bound that transfers $P(n\mid x)$'s (unbiased) length belief into the
  point estimate, cutting the residual MAP under-count while keeping $n=0$ at 0% (the
  floor only ever *raises* the bound). $\alpha\to0$ recovers the plain floored MAP;
  $\alpha\to$ median approaches a length-conditioned MAP at that quantile.
- **Posterior mean** — sensitive to the long right tail of the multiplicity posterior.
- **Posterior median** — the robust summary: it sidesteps both the MAP's mode
  collapse and the mean's tail sensitivity, giving the best multiplicity RMSE of the
  three. (On this 20-epoch checkpoint all three still over-count by ~1 node, so the
  plain-RSD baseline is closer here; fuller training is expected to remove the
  over-count.) For a *count*, prefer the posterior median.
- **MBR (perturbative Lund)** — the *mode-free, floor-free* estimate: the drawn tree
  of least expected Energy-Mover's-Distance to the posterior (`point_estimator="mbr"`,
  Komiske–Metodiev–Thaler EMD). Because an empty cloud pays the full mass-imbalance
  penalty, MBR **never collapses to $n=0$ with no floor** ($n=0$ stays 0% even at
  `min_emissions=0`), unlike the MAP — the brevity bias is removed *structurally* rather
  than clamped. Flip `MBR_BACKEND` to `energyflow` for the reference EMD (the same
  selected tree; its `.risk` differs by a $1/R$ scale). Needs the `[mbr]` extra.


In [ ]:
def jit(a, s=0.12):
    rng = np.random.default_rng(SEED)
    return np.asarray(a, float) + rng.uniform(-s, s, size=len(a))

estimators = [
    ("MAP",                 rec["n_map"],       C_MAP,     True),
    ("MAP (learned floor)", rec["n_map_floor"], "#ff7f0e", True),
    ("posterior mean",      rec["post_mean"],   C_POST,    False),
    ("posterior median",    rec["post_median"], "#9467bd", True),
    ("MBR (perturbative Lund)", rec["n_mbr"],   "#2ca02c", True),
]
lim = [-0.5, max(rec["n_true"].max(), max(e[1].max() for e in estimators)) + 1]

fig, axes = plt.subplots(1, 5, figsize=(25.5, 5.0), sharex=True, sharey=True)
for ax, (name, pred, col, do_jit) in zip(axes, estimators):
    ax.scatter(jit(rec["n_true"]), jit(rec["n_x"]), s=12, color=C_RSD, alpha=0.30,
               label=f"plain-RSD x  (bias {np.mean(rec['n_x']-rec['n_true']):+.2f})")
    ax.scatter(jit(rec["n_true"]), jit(pred) if do_jit else pred, s=18, color=col,
               alpha=0.6, label=f"{name}  (bias {np.mean(pred-rec['n_true']):+.2f})")
    ax.plot(lim, lim, "k--", lw=1, alpha=0.7)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(r"true multiplicity  $n_\mathrm{true}$")
    ax.set_title(name)
    ax.legend(loc="upper left", fontsize=9)

# the floored MAP (shown) never collapses; cite the un-floored fraction it avoids
frac0       = float(np.mean(rec["n_map"] == 0))       # floored (default) -> ~0
frac0_raw   = float(np.mean(rec["n_map_raw"] == 0))   # min_emissions=0   -> the collapse
axes[0].text(0.5, 0.045,
             f"floor min_emissions=1: {frac0*100:.0f}% at n=0\n"
             f"(un-floored MAP collapses for {frac0_raw*100:.0f}%)",
             transform=axes[0].transAxes, ha="center", va="bottom",
             fontsize=9, color=C_MAP,
             bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=C_MAP, alpha=0.9))
axes[0].set_ylabel("estimated multiplicity")
fig.suptitle("Multiplicity recovery vs truth — point estimators compared")
fig.tight_layout(); plt.show()

print("multiplicity bias ⟨n − n_true⟩ and RMSE:")
for name, pred, _, _ in [*estimators, ("MAP (un-floored)", rec["n_map_raw"], None, None),
                         ("plain-RSD x", rec["n_x"], None, None)]:
    bias = float(np.mean(pred - rec["n_true"]))
    rmse = float(np.sqrt(np.mean((pred - rec["n_true"]) ** 2)))
    print(f"  {name:18s}: bias {bias:+.3f}   RMSE {rmse:.3f}")
print(f"\nMAP n=0 collapse:  floored (min_emissions=1) = {frac0*100:.1f}%   "
      f"un-floored (min_emissions=0) = {frac0_raw*100:.1f}%")


### 6b. Leading-emission Lund distance to truth

Distance in the $(\ln 1/\Delta R,\ \ln k_t)$ plane between the *hardest* recovered
splitting and the truth's. The model's **posterior mode** should sit closer to the
truth than the plain-RSD baseline (mass left of grey).


In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 4.2))
hi = np.nanmax(np.concatenate([rec["d_mode"], rec["d_x"]]))
bins = np.linspace(0, hi + 1e-6, 28)
ax.hist(rec["d_x"],    bins=bins, color=C_RSD,  alpha=0.55, label=f"plain-RSD x  (mean {np.nanmean(rec['d_x']):.3f})")
ax.hist(rec["d_mode"], bins=bins, color=C_POST, alpha=0.70, label=f"posterior mode  (mean {np.nanmean(rec['d_mode']):.3f})")
ax.axvline(np.nanmean(rec["d_x"]),    color=C_RSD,  ls="--", lw=2)
ax.axvline(np.nanmean(rec["d_mode"]), color=C_POST, ls="--", lw=2)
ax.set_xlabel("leading-emission Lund distance to truth   (lower is better)")
ax.set_ylabel("jets")
ax.set_title("Leading-emission recovery")
ax.legend()
fig.tight_layout(); plt.show()


### 6c. Coordinate marginals — does the MAP reproduce the truth distributions?

Pooling every node across all jets, the MAP's groomed-observable marginals
($\ln k_t$, $\ln 1/\Delta R$, $\ln z$) should track the **truth** (green) more
faithfully than the plain-RSD hadron-level input (grey).


In [ ]:
MAP_ALL   = np.concatenate([c for c in map_coords if len(c)])   if any(len(c) for c in map_coords)   else np.zeros((0, 4))
TRUTH_ALL = np.concatenate([c for c in truth_coords if len(c)]) if any(len(c) for c in truth_coords) else np.zeros((0, 4))
X_ALL     = np.concatenate([c for c in x_coords if len(c)])     if any(len(c) for c in x_coords)     else np.zeros((0, 4))

specs = [(1, r"$\ln k_t$"), (0, r"$\ln 1/\Delta R$"), (2, r"$\ln z$")]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.0))
for ax, (col, lab) in zip(axes, specs):
    lo = min(TRUTH_ALL[:, col].min(), MAP_ALL[:, col].min(), X_ALL[:, col].min())
    hi = max(TRUTH_ALL[:, col].max(), MAP_ALL[:, col].max(), X_ALL[:, col].max())
    bins = np.linspace(lo, hi, 30)
    ax.hist(X_ALL[:, col],     bins=bins, density=True, histtype="stepfilled",
            color=C_RSD, alpha=0.35, label="plain-RSD x")
    ax.hist(TRUTH_ALL[:, col], bins=bins, density=True, histtype="step",
            color=C_TRUTH, lw=2.4, label="truth y")
    ax.hist(MAP_ALL[:, col],   bins=bins, density=True, histtype="step",
            color=C_MAP, lw=2.0, ls="--", label="MAP $\\hat y$")
    ax.set_xlabel(lab); ax.set_ylabel("density")
axes[0].legend(fontsize=9)
fig.suptitle("Groomed-observable marginals  (pooled over all nodes & jets)")
fig.tight_layout(); plt.show()


## 7. Posterior calibration — is the uncertainty trustworthy?

Conditional-generator posteriors are not automatically calibrated, so this is the
gate on whether the credible bands above can be believed.

- **SBC rank** of the true multiplicity among the draws should be **uniform**
  (Talts et al., [arXiv:1804.06788](https://arxiv.org/abs/1804.06788)).
- **PIT** should also be uniform with mean ≈ 0.5.
- The grey band is the expected ±1σ scatter for a perfectly uniform histogram.


In [ ]:
fig, (ar, ap) = plt.subplots(1, 2, figsize=(12.5, 4.3))
nb = 10
for ax, data, title in [(ar, rec["rank"], "SBC rank"), (ap, rec["pit"], "PIT")]:
    counts, _, _ = ax.hist(data, bins=nb, range=(0, 1), color=C_POST, alpha=0.8,
                           edgecolor="white")
    exp = len(data) / nb
    ax.axhline(exp, color="k", lw=1.5, label="uniform")
    ax.axhspan(exp - np.sqrt(exp), exp + np.sqrt(exp), color="k", alpha=0.12,
               label=r"$\pm 1\sigma$")
    ax.set_xlabel(f"{title}  (true multiplicity)")
    ax.set_ylabel("jets")
    ax.set_title(f"{title} uniformity   mean={np.mean(data):.3f}")
    ax.legend(fontsize=9)
chi2 = float(np.sum((np.histogram(rec["rank"], bins=nb, range=(0, 1))[0]
                     - len(rec["rank"]) / nb) ** 2 / (len(rec["rank"]) / nb)))
fig.suptitle(f"Simulation-based calibration   (SBC $\\chi^2$ = {chi2:.1f}, lower → more uniform)")
fig.tight_layout(); plt.show()


### 7a. Multiplicity coverage curve

For a grid of nominal credible levels, the fraction of jets whose **true**
multiplicity falls inside the posterior's central interval. A well-calibrated
posterior tracks the diagonal; below it ⇒ over-confident (too-narrow) posteriors.


In [ ]:
levels = np.linspace(0.05, 0.95, 19)
emp = []
for a in levels:
    lo_q, hi_q = (1 - a) / 2, (1 + a) / 2
    hits = [1.0 if np.quantile(m, lo_q) <= nt <= np.quantile(m, hi_q) else 0.0
            for m, nt in zip(post_mults, rec["n_true"])]
    emp.append(np.mean(hits))
emp = np.array(emp)

fig, ax = plt.subplots(figsize=(5.4, 5.2))
ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="perfect calibration")
ax.plot(levels, emp, "-o", color=C_POST, ms=4, label="empirical coverage")
ax.fill_between(levels, levels, emp, color=C_POST, alpha=0.12)
ax.set_xlabel("nominal credible level")
ax.set_ylabel("empirical coverage of truth")
ax.set_title("Multiplicity credible-interval coverage")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout(); plt.show()


## 8. Metrics summary

The same numbers the CLI prints (`h2p-rsd-junipr eval <ckpt>`), recovered from this
notebook's single evaluation pass. **Point estimate** = closer-to-truth / lower-bias
is better; **calibration** targets are coverage ≈ 0.68, SBC mean ≈ PIT mean ≈ 0.5,
SBC $\chi^2$ ≈ 0.


In [ ]:
summary = {
    "mean mult — truth y":            float(np.mean(rec["n_true"])),
    "mean mult — hadron x":           float(np.mean(rec["n_x"])),
    "mean mult — posterior":          float(np.mean(rec["post_mean"])),
    "dLund to truth — identity(x)":   float(np.nanmean(rec["d_x"])),
    "dLund to truth — posterior-mode": float(np.nanmean(rec["d_mode"])),
    "mult bias — identity(x)":        float(np.mean(rec["n_x"] - rec["n_true"])),
    "mult bias — MAP":                float(np.mean(rec["n_map"] - rec["n_true"])),
    "mult bias — posterior mean":     float(np.mean(rec["post_mean"] - rec["n_true"])),
    "mult bias — posterior median":   float(np.mean(rec["post_median"] - rec["n_true"])),
    "mult bias — MBR":                float(np.mean(rec["n_mbr"] - rec["n_true"])),
    "MAP n=0 frac (floored)":         float(np.mean(rec["n_map"] == 0)),
    "MAP n=0 frac (un-floored)":      float(np.mean(rec["n_map_raw"] == 0)),
    "MBR n=0 frac (floor-free)":      float(np.mean(rec["n_mbr"] == 0)),
    "leading-cell 68% coverage":      float(np.mean(rec["cover68"])),
    "SBC mean rank":                  float(np.mean(rec["rank"])),
    "SBC chi^2 (10 bins)":            chi2,
    "PIT mean":                       float(np.mean(rec["pit"])),
    "mean NLL/jet (test)":            float(nll.mean()),
}
w = max(len(k) for k in summary)
print(f"checkpoint: {ckpt.relative_to(REPO)}   |   source: {source_desc}")
print(f"jets evaluated: {len(rec['n_true'])}\n")
for k, v in summary.items():
    print(f"  {k:<{w}} : {v:+.3f}")


**Reading the figures**

- *Best point estimate* (§5, §6a–c): for the **leading-emission** observable the MAP /
  posterior-mode should beat the plain-RSD baseline, and the MAP marginals should
  overlay the truth marginals. For **multiplicity**, prefer the **posterior median** —
  the MAP is the *joint mode* and is length-biased; un-floored it collapses to $n=0$
  on the harder jets (§6a), which the `decode.min_emissions=1` floor removes, but the
  median is still the right summary for a count.
- *Posterior* (§5a, §7): credible bands should bracket the truth, the SBC/PIT
  histograms should be flat (mean ≈ 0.5), and the coverage curve should hug the
  diagonal. Coverage well below nominal ⇒ over-confident posteriors.

To run on **real** data, set `ROOT_PATH = "cpp/build/jets.root"` (or your own file)
and re-run. To evaluate a different model, set `CKPT_PATH`.
